# 08 ? Base model: with and without RAG

[Notebook guide](README.md) ? [Project overview](../README.md)

Run in a Colab T4 runtime, using the same workflow as notebook 05. This notebook loads **one model once**, then runs two test sections. The four sections across notebooks 07 and 08 share identical questions, references, prompt, 8-bit loading and decoding settings. No training is performed.

The setup uses the simple installation approach from notebooks 05/06, with additional metric packages; it does not repair every preinstalled Colab package. Run installation in a fresh runtime, restart the kernel if requested, then continue with imports. Packages are not pinned to a separately validated environment. The supplied Colab run completed; its answers, metrics, findings and remaining environment limitations are documented in [results/](../results/README.md).

Each section first evaluates the same four isolated questions from notebook 06, then provides the original notebook 05 conversation tests with their original history resets. Quantization stays fixed at 8 bits; its separate effect is not measured. The original strict-context prompt is copied verbatim: **empty context should produce refusal**, which is correct policy following even if reference-answer overlap is low.

For the main fine-tuning comparison, compare section 1 with section 4 on the isolated questions. Their retrieval uses empty history and the same index/settings. Conversation runs retain the original history-aware retriever: model-dependent reformulation can change retrieved passages, so those conversations compare the full pipeline. In the no-RAG branch, history is recorded but is not inserted into the answer prompt, matching the original answer prompt's inputs (`input` and `context`).


## Setup ? install dependencies


In [1]:
%pip install -q langchain langchain-classic langchain-community langchain-huggingface faiss-cpu transformers accelerate sentence-transformers bitsandbytes rouge-score nltk bert-score scikit-learn pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### Imports, Drive and model path


In [2]:
import warnings
import logging
warnings.filterwarnings("ignore")

import os
import time
import torch
import pandas as pd
from pathlib import Path
from google.colab import drive
from huggingface_hub import login
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score_fn
from sklearn.metrics.pairwise import cosine_similarity

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

warnings.filterwarnings("ignore")

# Suppress bitsandbytes and autograd warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Suppress Hugging Face messages
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
drive.mount('/content/drive')
login()  # Use your own account with access to the gated base model.


Mounted at /content/drive


In [4]:
db_path = '/content/drive/MyDrive/NLP_proyect/vector_database'
model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
output_dir = Path('/content/drive/MyDrive/NLP_proyect/evaluation_simple/base')
output_dir.mkdir(parents=True, exist_ok=True)


### Original 8-bit model loading and generation pipeline


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [6]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.15,
    do_sample=False,
    return_full_text=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)


### Retrieval, original prompt and chains


In [7]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2", model_kwargs={"device": "cpu"})
vector_db = FAISS.load_local(db_path, embedding_model, allow_dangerous_deserialization=True)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The embedding model runs on CPU to leave T4 memory for the generator. The checkpoint, search depth and default embedding normalization remain the same. Only load the saved pickle index from your own trusted files.


In [8]:
contextualize_q_system_prompt = """Ukoliko postoji istorija razgovora i korisnik postavi pitanje koje se nadovezuje,
reformuliši to pitanje tako da bude samostalno i razumljivo bez istorije razgovora.
NEMOJ odgovarati na pitanje, samo ga reformuliši ako je potrebno, u suprotnom ga vrati identičnog."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

system_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
1. Ti si stručni medicinski asistent i komuniciraš isključivo na srpskom jeziku.
2. Ako u kontekstu nema informacija potrebnih za odgovor, nemoj izmišljati nego jasno reci: "Nažalost, nemam te informacije u svojoj bazi podataka, najbolje se savetuj sa lekarom ili farmaceutom. Informacije sa interneta mogu biti lažne i štetne."
3. Piši svakodnevnim i lako shvatljivim jezikom.
4. Ukoliko je pitanje na latinici, odgovori na latinici. Ukoliko je pitanje na ćirilici, odgovori na ćirilici.
5. Na kraju svakog odgovora OBAVEZNO dodaj: "Pre upotrebe svakog medikamenta, obavezno se posavetuje sa lekarom ili farmaceutom. Dobijeni odgovor nije zamena za stručno mišljenje, već pomoćna alatka."

Pruženi medicinski kontekst:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

qa_prompt = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "input"]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [9]:
# The RAG branch is the original chain from notebook 05.
# The other branch uses the same prompt and LLM with an empty context.
no_rag_chain = qa_prompt | llm

def pitaj(pitanje):
    t0 = time.time()
    if koristi_rag:
        rezultat = rag_chain.invoke({'input': pitanje, 'chat_history': chat_history})
        odgovor = rezultat['answer']
        dokumenti = rezultat.get('context', [])
    else:
        odgovor = no_rag_chain.invoke({'input': pitanje, 'context': ''})
        dokumenti = []
    trajanje = time.time() - t0
    chat_history.append(HumanMessage(content=pitanje))
    chat_history.append(AIMessage(content=odgovor))
    print(f'Q: {pitanje}\nA: {odgovor}\n[{trajanje:.1f}s]\n')
    return odgovor, dokumenti, trajanje


### Shared reference cases and metric definitions


Questions, reference strings (including existing typos), and metric definitions are copied from notebook 06. This four-question sample is a pilot. The original ROUGE tokenizer has limitations for Cyrillic; interpret those scores accordingly. Metrics measure reference similarity and do not establish medical correctness. BERTScore runs on CPU, avoiding a second transformer on the T4.


In [10]:
# TODO: populate referentni_odgovor with a reference answer from the knowledge base
# (a short, factually correct answer expected from the model).

eval_dataset_odgovori = [
    {
        "pitanje": "Kada ne smem da uzimam lek Dovicin?",
        "referentni_odgovor": '''Lek Dovicin ne smete uzimati:
- ukoliko ste alergični (preosetljivi) na doksiciklin ili neki drugi lek iz grupe tetraciklina ili na bilo koju od od pomoćnih supstanci ovog leka (navedene u odeljku 6);
- ukoliko ste trudni ili planirate trudnoću;
- ukoliko dojite;
- ukoliko imate oboljenje praćeno otežanim gutanjem.
Lek Dovicin ne treba koristiti tokom perioda razvoja zuba (u periodu trudnoće, kod odojčadi i dece mlađe od 8 godina)
''',
    },
    {
        "pitanje": "Koja je uobičajena doza za lek Nimulid?",
        "referentni_odgovor": "Lek Nimulid uvek uzimajte tačno onako kako Vam je objasnio Vaš lekar. Ukoliko niste sigurni, proverite sa Vašim lekarom ili farmaceutom. Uobičajena doza leka je jedna tableta (100 mg) dva puta dnevno, posle obroka.",
    },
    {
        "pitanje": "Da li deca smeju da konzumiraju Pressing?",
        "referentni_odgovor": "Deca: Nemojre davati lel Pressing sirup deci mlađoj od 2 godine.",
    },
    {
        "pitanje": "Za šta se primenjuje lek Zoloft",
        "referentni_odgovor": '''Lek Zoloft se može primenjivati za lečenje:
• Depresije i prevenciju ponovnog javljanja depresije (kod odraslih).
• Socijalnog anksioznog poremećaja (kod odraslih).
• Posttraumatskog stresnog poremećaja (kod odraslih).
• Paničnog poremećaja (kod odraslih).
• Opsesivno-kompulzivnog poremećaja (kod odraslih i dece i adolescenata uzrasta od 6 do 17 godina).''',
    },

]

len(eval_dataset_odgovori)


4

In [11]:
rouge_l_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
smoothing = SmoothingFunction().method1


def racunaj_rouge_l(generisan, referentan):
    rezultat = rouge_l_scorer.score(referentan, generisan)
    return rezultat["rougeL"].fmeasure


def racunaj_bleu(generisan, referentan):
    referenca_tok = [referentan.split()]
    kandidat_tok = generisan.split()
    return sentence_bleu(referenca_tok, kandidat_tok, smoothing_function=smoothing)


def racunaj_cos_similarity(generisan, referentan):
    emb_gen = embedding_model.embed_query(generisan)
    emb_ref = embedding_model.embed_query(referentan)
    sim = cosine_similarity([emb_gen], [emb_ref])[0][0]
    return float(sim)


## Test 3 ? Base model without RAG

Run this section from the first cell. The test only changes the selected chain; the model, prompt and decoding settings are shared with the other section. Four isolated questions use empty history individually. Generated answers, retrieved text and metrics are saved as CSV files on Drive, overwriting earlier files for this section when rerun. Save the notebook to retain conversation outputs too.


In [12]:
koristi_rag = False
test_name = 'test_3_base_no_rag'
chat_history = []


In [13]:
# Same four independent questions in all four tests.
generisani_odgovori = []
redovi = []
for primer in eval_dataset_odgovori:
    chat_history = []
    odgovor, dokumenti, trajanje = pitaj(primer['pitanje'])
    generisani_odgovori.append(odgovor)
    redovi.append({
        'Pitanje': primer['pitanje'],
        'Odgovor': odgovor,
        'Referentni odgovor': primer['referentni_odgovor'],
        'Vreme (s)': trajanje,
        'Izvori': '; '.join(str(d.metadata.get('izvor_fajl', '')) for d in dokumenti),
        'Kontekst': '\n\n'.join(d.page_content for d in dokumenti),
    })
# Save answers before calculating metrics, so a metric error does not lose them.
df_odgovori = pd.DataFrame(redovi)
df_odgovori.to_csv(output_dir / f'{test_name}_answers.csv', index=False)
display(df_odgovori[['Pitanje', 'Odgovor', 'Vreme (s)', 'Izvori']])


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: Kada ne smem da uzimam lek Dovicin?
A: 

Dovicin (doksiciklin) je antibiotik koji se koristi za lečenje infekcija uzrokovanih bakterija. Preporučeno je da se uzima po preporuci lekara ili farmaceuta.

Ako ste zadržali ovaj lek bez resepte ili ga uzimali prema samom receptu, morate da se obratite lekaru ili farmaceutu kako bi se utvrdilo da li ste u stanju da nastavite sa uzimanjem leka.

Nije mi poznato da li ste uzeli ovaj leka bez resepte ili da li ste nastavili sa uzimanje leka nakon što je istekao rok trajanja. Zbog toga, ne mogu da vam dajem precizan odgovor.

Pre upotrebu svakog leka, obavezna je konzultacija sa lekar ili farmaceut.
[39.6s]

Q: Koja je uobičajena doza za lek Nimulid?
A: 

Nimulid je lek koji sadrži indometacin, koji je nesteroidni antiinflamatorni lek (NSAID). Uobičajene doze za Nimulid variraju ovisno o bolesti i stanju za koje se koristi.

U slučaju bolova u reumatskim bolestima, uobičajan početak terapije je 25-50 mg dvaput dnevno, a maksimalna doza je 150 

,Pitanje,Odgovor,Vreme (s),Izvori
0,Kada ne smem da uzimam lek Dovicin?,\n\nDovicin (doksiciklin) je antibiotik koji s...,39.606959,
1,Koja je uobičajena doza za lek Nimulid?,"\n\nNimulid je lek koji sadrži indometacin, ko...",41.880667,
2,Da li deca smeju da konzumiraju Pressing?,"\n\nNe, deci ne smeju da konsumiraju Pressin. ...",34.268389,
3,Za šta se primenjuje lek Zoloft,\n\nLek Zoloft (sertralin) je selektivni inhib...,79.214932,


In [14]:
referentni_odgovori = [p['referentni_odgovor'] for p in eval_dataset_odgovori]
bert_p, bert_r, bert_f1 = bert_score_fn(
    generisani_odgovori, referentni_odgovori,
    lang='sr', device='cpu', batch_size=1, verbose=False,
)
rezultati_odgovora = []
for i, primer in enumerate(eval_dataset_odgovori):
    gen = generisani_odgovori[i]
    ref = primer['referentni_odgovor']
    rezultati_odgovora.append({
        'Pitanje': primer['pitanje'],
        'ROUGE-L': racunaj_rouge_l(gen, ref),
        'BLEU': racunaj_bleu(gen, ref),
        'BERTScore (F1)': bert_f1[i].item(),
        'Cosine similarity': racunaj_cos_similarity(gen, ref),
        'Vreme (s)': redovi[i]['Vreme (s)'],
    })
df_metrike = pd.DataFrame(rezultati_odgovora)
display(df_metrike)
display(df_metrike.drop(columns='Pitanje').mean().to_frame('PROSEK'))
df_metrike.to_csv(output_dir / f'{test_name}_metrics.csv', index=False)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity,Vreme (s)
0,Kada ne smem da uzimam lek Dovicin?,0.101124,0.003178,0.642470,0.635312,39.606959
1,Koja je uobičajena doza za lek Nimulid?,0.113924,0.002742,0.673534,0.709741,41.880667
2,Da li deca smeju da konzumiraju Pressing?,0.016529,0.001843,0.618122,0.689566,34.268389
3,Za šta se primenjuje lek Zoloft,0.125984,0.011045,0.726832,0.834047,79.214932


,PROSEK
ROUGE-L,0.089390
BLEU,0.004702
BERTScore (F1),0.665239
Cosine similarity,0.717166
Vreme (s),48.742737


### Optional ? original conversation tests from notebook 05

These calls and the two history resets are copied in the original order. They are separate from the four isolated questions above. Run the following cells only if you also want to inspect the original multi-turn examples; they do not contribute to the metric table.


##### Session A: Dovicin in Latin script

**History reset:** the next cell starts a new conversation.

In [15]:
chat_history = []


In [16]:
pitaj("Kada ne smem da uzimam lek Dovicin?")

Q: Kada ne smem da uzimam lek Dovicin?
A: 

Dovicin (doksiciklin) je antibiotik koji se koristi za lečenje infekcija uzrokovanih bakterija. Preporučeno je da se uzima po preporuci lekara ili farmaceuta.

Ako ste zadržali ovaj lek bez resepte ili ga uzimali prema samom receptu, morate da se obratite lekaru ili farmaceutu kako bi se utvrdilo da li ste u stanju da nastavite sa uzimanjem leka.

Nije mi poznato da li ste uzeli ovaj leka bez resepte ili da li ste nastavili sa uzimanje leka nakon što je istekao rok trajanja. Zbog toga, ne mogu da vam dajem precizan odgovor.

Pre upotrebu svakog leka, obavezna je konzultacija sa lekar ili farmaceut.
[31.1s]



('\n\nDovicin (doksiciklin) je antibiotik koji se koristi za lečenje infekcija uzrokovanih bakterija. Preporučeno je da se uzima po preporuci lekara ili farmaceuta.\n\nAko ste zadržali ovaj lek bez resepte ili ga uzimali prema samom receptu, morate da se obratite lekaru ili farmaceutu kako bi se utvrdilo da li ste u stanju da nastavite sa uzimanjem leka.\n\nNije mi poznato da li ste uzeli ovaj leka bez resepte ili da li ste nastavili sa uzimanje leka nakon što je istekao rok trajanja. Zbog toga, ne mogu da vam dajem precizan odgovor.\n\nPre upotrebu svakog leka, obavezna je konzultacija sa lekar ili farmaceut.',
 [],
 31.0887451171875)

In [17]:
pitaj("A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?")

Q: A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?
A: 

Ne mogu da vam pruim informacije o tome kako da koristite Dovicin zajedno sa drugim lekovima. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutom.
[9.6s]



('\n\nNe mogu da vam pruim informacije o tome kako da koristite Dovicin zajedno sa drugim lekovima. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutom.',
 [],
 9.589040756225586)

#### Session A continued: Dovicin in Cyrillic

The original notebook does not reset history here. Earlier turns remain available.

In [18]:
pitaj("Када не смем да узимам лек Dovicin?")

Q: Када не смем да узимам лек Dovicin?
A: 

Dovicin (doksiciklin) je antibiotik koji se koristi za lečenje infekcija uzrokovanih bakterijama. Preporučeno je da se uzima po receptu.

Ako ste primili recept za Dovicin, morate ga uzeti kako ste uputio vam lekar ili farmacevt. Obično se uzima dvaput dnevno, bez obzira na obrok.

Nemojte prestajati uzimanju Dovicina bez saveta lekara ili farmacevta, jer to može dovesti do pogoršanja bolesti ili razvoja otpornosti bakterija na lek.

Ako imate neki od sledećih simptoma, obavestite liječnika ili farmaceuta:

- Teško dišanje
- Bolest u grlu ili grlo
- Boli u stomaku ili trbuhu
- Gubitak apetita
- Mučnina ili povraćanje
- Suha usta ili sluzokoža
- Bolestan ili oslabljen
- Bilo koji drugi neželjeni efekt

Pre upotrebu svakog leka, obavezna se posavjetuje sa liječnikom ili farmaceutom. Dobijen odgovor ne zamjenjuje stručno mislenje, vece pomocna alatika.
[48.6s]



('\n\nDovicin (doksiciklin) je antibiotik koji se koristi za lečenje infekcija uzrokovanih bakterijama. Preporučeno je da se uzima po receptu.\n\nAko ste primili recept za Dovicin, morate ga uzeti kako ste uputio vam lekar ili farmacevt. Obično se uzima dvaput dnevno, bez obzira na obrok.\n\nNemojte prestajati uzimanju Dovicina bez saveta lekara ili farmacevta, jer to može dovesti do pogoršanja bolesti ili razvoja otpornosti bakterija na lek.\n\nAko imate neki od sledećih simptoma, obavestite liječnika ili farmaceuta:\n\n- Teško dišanje\n- Bolest u grlu ili grlo\n- Boli u stomaku ili trbuhu\n- Gubitak apetita\n- Mučnina ili povraćanje\n- Suha usta ili sluzokoža\n- Bolestan ili oslabljen\n- Bilo koji drugi neželjeni efekt\n\nPre upotrebu svakog leka, obavezna se posavjetuje sa liječnikom ili farmaceutom. Dobijen odgovor ne zamjenjuje stručno mislenje, vece pomocna alatika.',
 [],
 48.64341640472412)

In [19]:
pitaj("Зашто си ми генерисао одговор на латиници?")

Q: Зашто си ми генерисао одговор на латиници?
A: 

Žao mi je što sam vam generisao odgovor na latinici umesto na srpskome jeziku. Kao stručni medicijski asistent, trebaću da nastavim da komuniciram na srpskim. Evo ponovnog odgovor:

Zbog čega ste mi generisali odgovor u latinici?
[12.8s]



('\n\nŽao mi je što sam vam generisao odgovor na latinici umesto na srpskome jeziku. Kao stručni medicijski asistent, trebaću da nastavim da komuniciram na srpskim. Evo ponovnog odgovor:\n\nZbog čega ste mi generisali odgovor u latinici?',
 [],
 12.772185325622559)

In [20]:
pitaj("Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.")

Q: Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.
A: 

Извини, али не могу да одговорим на то питање. Ако имате неку другу ствар, можда могу да вам помогнем.
[6.2s]



('\n\nИзвини, али не могу да одговорим на то питање. Ако имате неку другу ствар, можда могу да вам помогнем.',
 [],
 6.217999458312988)

#### Session A continued: Nimulid

This block continues the same history; it is not an isolated test.

In [21]:
pitaj("Која је уобичајена доза за лек Nimulid?")

Q: Која је уобичајена доза за лек Nimulid?
A: 

Nimulid (Suhacetin) se koristi za liječenje bolova u reumatskim bolestima.

Uobičajena doza za odrasle osobe je 100-200 mg 2 puta dnevno, maksimalno 400 mg dnevno.

Pre upotrebu svakog leka, obavezna se posavjetuje sa lekara ili farmaceuta. Dobijen odgovor nisi zamena za struktno misljenje vec pomocna alatika.
[18.2s]



('\n\nNimulid (Suhacetin) se koristi za liječenje bolova u reumatskim bolestima.\n\nUobičajena doza za odrasle osobe je 100-200 mg 2 puta dnevno, maksimalno 400 mg dnevno.\n\nPre upotrebu svakog leka, obavezna se posavjetuje sa lekara ili farmaceuta. Dobijen odgovor nisi zamena za struktno misljenje vec pomocna alatika.',
 [],
 18.163453102111816)

In [22]:
pitaj("Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q: Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?
A: 

Ne može mi da objasnim detaljno šta bi se moglo desiti ako ste prekorčili doziranje tablete. Međutim, mogu vam reći da je važno da se što prije obratite liječniku ili farmaceutu jer oni će vam dati savjet koji će vam olakšati situaciju.

Uopće, preporučeno je da se tablete uzme po dozama koje su navedene na etiketi ili koje su određene od strane liječnika. Prekoracenje doze može dovesti do neželjenih efekata ili povećanja rizika od neželjenog stanja ili bolesti.

Ako ste uzeli više tableta, molim vas da se obratite stručnjaku što prije.
[29.5s]



('\n\nNe može mi da objasnim detaljno šta bi se moglo desiti ako ste prekorčili doziranje tablete. Međutim, mogu vam reći da je važno da se što prije obratite liječniku ili farmaceutu jer oni će vam dati savjet koji će vam olakšati situaciju.\n\nUopće, preporučeno je da se tablete uzme po dozama koje su navedene na etiketi ili koje su određene od strane liječnika. Prekoracenje doze može dovesti do neželjenih efekata ili povećanja rizika od neželjenog stanja ili bolesti.\n\nAko ste uzeli više tableta, molim vas da se obratite stručnjaku što prije.',
 [],
 29.5058536529541)

In [23]:
pitaj("Такође сам конзумирао алкохол. Да ли ми се нешто може десити?")

Q: Такође сам конзумирао алкохол. Да ли ми се нешто може десити?
A: 

Ako ste konzumirali alkohol, moguće su neke neželene reakcije, kao što su glavobolja, mučnina, povraćanje, osip, vrtoglavljenje, promene u raspoloženju, teško spavanje, teški snos, slabost, gubitak apetita, poremećaji u funkciji jetre, poremeće se regulacija nivoa šećera u krvi, poremeče se regulacija krvnih grločica, poremečena regulacija naponova u mozgu, poremece se funkcije bubrega, poremeči se metabolizam, poremeći se nivo hormona, poremečen rad jetre, promene nivoa enzima, poremečan rad bubrega, promene funkcija jetre, teške promene u funkciji bubrega, tešci poremećaj u funkciji jetera, poremechanje funkcije bubreg, poremeche se funkcije jetre, poremećen rad bubrega.

Ako ste primili veliku doziranje alkohola ili ste konzumovali alkohol tokom duže vremena, mogu se javiti i ozbiljnije zdravstvene probleme kao što su:

- hepatitisa (zapaljenje jetre)
- cirhroza (zadebljanje jetre zbog oštećenja tkiva)
- alkoholna

('\n\nAko ste konzumirali alkohol, moguće su neke neželene reakcije, kao što su glavobolja, mučnina, povraćanje, osip, vrtoglavljenje, promene u raspoloženju, teško spavanje, teški snos, slabost, gubitak apetita, poremećaji u funkciji jetre, poremeće se regulacija nivoa šećera u krvi, poremeče se regulacija krvnih grločica, poremečena regulacija naponova u mozgu, poremece se funkcije bubrega, poremeči se metabolizam, poremeći se nivo hormona, poremečen rad jetre, promene nivoa enzima, poremečan rad bubrega, promene funkcija jetre, teške promene u funkciji bubrega, tešci poremećaj u funkciji jetera, poremechanje funkcije bubreg, poremeche se funkcije jetre, poremećen rad bubrega.\n\nAko ste primili veliku doziranje alkohola ili ste konzumovali alkohol tokom duže vremena, mogu se javiti i ozbiljnije zdravstvene probleme kao što su:\n\n- hepatitisa (zapaljenje jetre)\n- cirhroza (zadebljanje jetre zbog oštećenja tkiva)\n- alkoholna neuropatija (oštećenje živaca)\n- alkanski sindrom (zbog 

##### Session B: Pressing

**History reset:** the next cell starts a new conversation.

In [24]:
chat_history = []


In [25]:
pitaj("Da li deca smeju da konzumiraju Pressing?")

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Ne, deci ne smeju da konsumiraju Pressin. Preporučeno je da ga koriste samo odrasli koji imaju dijagnozu hipertenzije ili drugih kardiovaskularnih bolesti, pod nadzorom lekara.

Pressin sadrži aktivne sastojke koji mogu da budu štetni za zdravlje dece. Oni mogu da utiču na razvoj hipertenzija kod dece, kao i da pogoršaju postojeće hipertenziju.

Deca treba da konzumuju hranu koja je bogata vitaminima i mineralima, kao što su voće, povrće, zito-bobice, mleko i mlečni proizvodi. Ako ste zabrinuti za zdravstveni status vašeg deteta, savetujte se sa pedijetrom ili lekarom koji će vam dati savete o najboljoj hrani za vaše dete.
[34.4s]



('\n\nNe, deci ne smeju da konsumiraju Pressin. Preporučeno je da ga koriste samo odrasli koji imaju dijagnozu hipertenzije ili drugih kardiovaskularnih bolesti, pod nadzorom lekara.\n\nPressin sadrži aktivne sastojke koji mogu da budu štetni za zdravlje dece. Oni mogu da utiču na razvoj hipertenzija kod dece, kao i da pogoršaju postojeće hipertenziju.\n\nDeca treba da konzumuju hranu koja je bogata vitaminima i mineralima, kao što su voće, povrće, zito-bobice, mleko i mlečni proizvodi. Ako ste zabrinuti za zdravstveni status vašeg deteta, savetujte se sa pedijetrom ili lekarom koji će vam dati savete o najboljoj hrani za vaše dete.',
 [],
 34.3852162361145)

In [26]:
pitaj("Da li odrasle osobe mogu da konzumiraju Pressing?")

Q: Da li odrasle osobe mogu da konzumiraju Pressing?
A: 

Ne, odrasli ne bi trebalo da konzumuju Pressing. Pressing je lijek koji sadrži fenobarbital, koji je barbiturat, a njegova upotreba kod odraslog stanovništva nije dozvoljena zbog mogućnosti vikljuštva i drugih neželjenih efekata.

Prema dostupnim informacijama, Pressing je rezerviran za djecu od 3 godine i starije, kao sredstvo za sprečavanje epileptičkih napada.
[19.9s]



('\n\nNe, odrasli ne bi trebalo da konzumuju Pressing. Pressing je lijek koji sadrži fenobarbital, koji je barbiturat, a njegova upotreba kod odraslog stanovništva nije dozvoljena zbog mogućnosti vikljuštva i drugih neželjenih efekata.\n\nPrema dostupnim informacijama, Pressing je rezerviran za djecu od 3 godine i starije, kao sredstvo za sprečavanje epileptičkih napada.',
 [],
 19.915151119232178)

In [27]:
pitaj("Ali ja te ne pitam za lek Crestora, već za lek Pressing?")

Q: Ali ja te ne pitam za lek Crestora, već za lek Pressing?
A: 

Ne mogu da vam pruim informacije o leku Pressing. Pre upotrebe bilo kog medikamnta, obavezan se posaveti sa lekarim ili farmaceutom. Dobijen odgovor nisu zamena za strucno misljenje vec pomocna alatika.
[11.5s]



('\n\nNe mogu da vam pruim informacije o leku Pressing. Pre upotrebe bilo kog medikamnta, obavezan se posaveti sa lekarim ili farmaceutom. Dobijen odgovor nisu zamena za strucno misljenje vec pomocna alatika.',
 [],
 11.510277032852173)

#### Session B continued: more specific wording

History is retained while the wording of the question changes.

In [28]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca ne smeju da konsumiraju Pressin sirup bez prethodnog saveta lekara ili farmaceuta. Pressin sirupe sadrže aktivne sastojke koji mogu bitno uticati na zdravlje dece.

Preporučeno je da se kod dece koriste specijalno dizajnirani sirupi koji su bezbedni za njihovo uzrast. Ako ste zainteresovani za zamenu za Pressin sir up, savetujte se sa lekarjem ili farmaceutom kako bi se pronašla adekvatna alternativa.

Pre upotrebu svakog leka, obavezna je konzultacija sa lekarima ili farmaceutima.
[25.0s]



('\n\nNe, deca ne smeju da konsumiraju Pressin sirup bez prethodnog saveta lekara ili farmaceuta. Pressin sirupe sadrže aktivne sastojke koji mogu bitno uticati na zdravlje dece.\n\nPreporučeno je da se kod dece koriste specijalno dizajnirani sirupi koji su bezbedni za njihovo uzrast. Ako ste zainteresovani za zamenu za Pressin sir up, savetujte se sa lekarjem ili farmaceutom kako bi se pronašla adekvatna alternativa.\n\nPre upotrebu svakog leka, obavezna je konzultacija sa lekarima ili farmaceutima.',
 [],
 24.96598982810974)

In [29]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je kombinovani preparat koji sadrži paracetamol i fenilefrin hidrohlorid. On se koristi za ublažavanje bolova i smanjenje upale.

Paracetamol je analgetik i antipiretik koji deluje kao centralni agens koji smanjuje proizvodnju prostaglandina, što dovodi do smanjenja bola i temperature tela. Fenilefrin je vazokonstriktor koji smanji protok krvi u tkivima, što može da smanji otok i bol.

Preporučeno je da se Pressing sirupa koristi samo po receptu, a uzimanje prema receptu je obavezno. Prema receptu, Pressing sirap se može koristiti za tretiranje bolova u slučajevima kao što su:

* Bolovi u stomaku (kolika, dispepsija)
* Bolovi glave
* Bolovi grla
* Bol u rektumu
* Menstrualni bolovi

Prema receptu treba uzeti 1-2 čajne kašike (5-10 ml) 3-4 puta dnevno. Pre upotrebe, morate konzultovati liječnika ili farmaceuta.
[45.3s]



('\n\nPressing sirup je kombinovani preparat koji sadrži paracetamol i fenilefrin hidrohlorid. On se koristi za ublažavanje bolova i smanjenje upale.\n\nParacetamol je analgetik i antipiretik koji deluje kao centralni agens koji smanjuje proizvodnju prostaglandina, što dovodi do smanjenja bola i temperature tela. Fenilefrin je vazokonstriktor koji smanji protok krvi u tkivima, što može da smanji otok i bol.\n\nPreporučeno je da se Pressing sirupa koristi samo po receptu, a uzimanje prema receptu je obavezno. Prema receptu, Pressing sirap se može koristiti za tretiranje bolova u slučajevima kao što su:\n\n* Bolovi u stomaku (kolika, dispepsija)\n* Bolovi glave\n* Bolovi grla\n* Bol u rektumu\n* Menstrualni bolovi\n\nPrema receptu treba uzeti 1-2 čajne kašike (5-10 ml) 3-4 puta dnevno. Pre upotrebe, morate konzultovati liječnika ili farmaceuta.',
 [],
 45.25223183631897)

In [30]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca ne smeju da konsumiraju Pressin sirup bez prethodnog saveta lekara ili farmaceuta. Pressin sirupe sadrže aktivne sastojke koji mogu bitno uticati na zdravlje dece.

Preporučeno je da se kod dece koriste specijalno dizajnirani sirupi koji su bezbedni za njihovo uzrast. Ako ste zainteresovani za zamenu za Pressin sir up, savetujte se sa lekarjem ili farmaceutom kako bi se pronašla adekvatna alternativa.

Pre upotrebu svakog leka, obavezna je konzultacija sa lekarima ili farmaceutima.
[24.9s]



('\n\nNe, deca ne smeju da konsumiraju Pressin sirup bez prethodnog saveta lekara ili farmaceuta. Pressin sirupe sadrže aktivne sastojke koji mogu bitno uticati na zdravlje dece.\n\nPreporučeno je da se kod dece koriste specijalno dizajnirani sirupi koji su bezbedni za njihovo uzrast. Ako ste zainteresovani za zamenu za Pressin sir up, savetujte se sa lekarjem ili farmaceutom kako bi se pronašla adekvatna alternativa.\n\nPre upotrebu svakog leka, obavezna je konzultacija sa lekarima ili farmaceutima.',
 [],
 24.860896110534668)

In [31]:
pitaj("A deca starija od 2 godine?")

Q: A deca starija od 2 godine?
A: 

Deca starija 2 godina mogu da uzimaju različite vrste tableta ili kapsula, ali samo ako su odobreni od strane lekara ili pedijatra.

U slučaju da ste tražili informacije o tabletimima ili kapsulama koje se mogu dati dečaci starijima od 2, neki primeri su:

- Paracetamol (acetaminofen) - može se dati dečacima starijim od 2 meseca, ali samo u obliku sirupa ili čvrste forme, a doziranje treba da bude određeno od strane pedijatre.
- Ibuprofen - može da se daje dečacim starijih od 6 meseci, ali samo po receptu i pod nadzorom pedijatre.

Mnoga druga lekova, kao što su aspirin ili kodein, ne smeju da se daju dečacem starijma od 2 ili 3 godine zbog rizika od oštećenja jetre ili zavisnosti.

Pre upotrebu svakog leka, obavezna je konzultacija sa lekarjem ili pedijatrom koji će vam dati savete o najprikladnijoj terapiji za vaše dete.
[45.7s]



('\n\nDeca starija 2 godina mogu da uzimaju različite vrste tableta ili kapsula, ali samo ako su odobreni od strane lekara ili pedijatra.\n\nU slučaju da ste tražili informacije o tabletimima ili kapsulama koje se mogu dati dečaci starijima od 2, neki primeri su:\n\n- Paracetamol (acetaminofen) - može se dati dečacima starijim od 2 meseca, ali samo u obliku sirupa ili čvrste forme, a doziranje treba da bude određeno od strane pedijatre.\n- Ibuprofen - može da se daje dečacim starijih od 6 meseci, ali samo po receptu i pod nadzorom pedijatre.\n\nMnoga druga lekova, kao što su aspirin ili kodein, ne smeju da se daju dečacem starijma od 2 ili 3 godine zbog rizika od oštećenja jetre ili zavisnosti.\n\nPre upotrebu svakog leka, obavezna je konzultacija sa lekarjem ili pedijatrom koji će vam dati savete o najprikladnijoj terapiji za vaše dete.',
 [],
 45.683250427246094)

#### Session B continued: repeated standalone wording

Although the question is self-contained, the original code does not reset conversation history.

In [32]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Pressing sirup je kombinovani preparat koji sadrži paracetamol i fenilefrin hidrohlorid. On se koristi za ublažavanje bolova i smanjenje upale.

Paracetamol je analgetik i antipiretik koji deluje kao centralni agens koji smanjuje proizvodnju prostaglandina, što dovodi do smanjenja bola i temperature tela. Fenilefrin je vazokonstriktor koji smanji protok krvi u tkivima, što može da smanji otok i bol.

Preporučeno je da se Pressing sirupa koristi samo po receptu, a uzimanje prema receptu je obavezno. Prema receptu, Pressing sirap se može koristiti za tretiranje bolova u slučajevima kao što su:

* Bolovi u stomaku (kolika, dispepsija)
* Bolovi glave
* Bolovi grla
* Bol u rektumu
* Menstrualni bolovi

Prema receptu treba uzeti 1-2 čajne kašike (5-10 ml) 3-4 puta dnevno. Pre upotrebe, morate konzultovati liječnika ili farmaceuta.
[45.5s]



('\n\nPressing sirup je kombinovani preparat koji sadrži paracetamol i fenilefrin hidrohlorid. On se koristi za ublažavanje bolova i smanjenje upale.\n\nParacetamol je analgetik i antipiretik koji deluje kao centralni agens koji smanjuje proizvodnju prostaglandina, što dovodi do smanjenja bola i temperature tela. Fenilefrin je vazokonstriktor koji smanji protok krvi u tkivima, što može da smanji otok i bol.\n\nPreporučeno je da se Pressing sirupa koristi samo po receptu, a uzimanje prema receptu je obavezno. Prema receptu, Pressing sirap se može koristiti za tretiranje bolova u slučajevima kao što su:\n\n* Bolovi u stomaku (kolika, dispepsija)\n* Bolovi glave\n* Bolovi grla\n* Bol u rektumu\n* Menstrualni bolovi\n\nPrema receptu treba uzeti 1-2 čajne kašike (5-10 ml) 3-4 puta dnevno. Pre upotrebe, morate konzultovati liječnika ili farmaceuta.',
 [],
 45.548418045043945)

#### Session B continued: Zoloft

The existing history remains in use.

In [33]:
pitaj("Za šta se primenjuje lek Zoloft")

Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zoloft (sertralin) je selektivni inhibitor reuptake serotonina (SSRI), koji se koristi za tretman depresije, anksioznih poremećaja, panike, fobija, obsesivno-kompulzivnog poremećaj, posttraumatske stresne poremećaje (PTSD) i premenstrualnog disforičnog poremenca (PMD).

Zoloft deluje tako što sprečava ponovno ugradnju serotonina u neuronske stanice, što dovodi do povećanja koncentracije serotonina dostupnog u sinapsama. To može poboljšati stanje pacijenta i smanjiti simptome bolesti.

Preporučeno je da se Zoloft uzima jednom dnevno, prije ili nakon obroka, ili kako je liječnik preporučio. Lek se može uzeti bez obrezivanja tablete ili u obliku kapsule.

Prema uputstvu za upotrebu, Zoloft se može koristiti kod odraslih i dece od 6 godina i starijih za tretiranje:

* Depresije
* Anksioznog poremeћaja
* Panike
* Fobije
* Obsesivno kompulzivni poremećaji
* Posttraumatski stresni poremeće
* Premenstrualni disforični poremeћaj

Međutim, važno je nap

('\n\nLek Zoloft (sertralin) je selektivni inhibitor reuptake serotonina (SSRI), koji se koristi za tretman depresije, anksioznih poremećaja, panike, fobija, obsesivno-kompulzivnog poremećaj, posttraumatske stresne poremećaje (PTSD) i premenstrualnog disforičnog poremenca (PMD).\n\nZoloft deluje tako što sprečava ponovno ugradnju serotonina u neuronske stanice, što dovodi do povećanja koncentracije serotonina dostupnog u sinapsama. To može poboljšati stanje pacijenta i smanjiti simptome bolesti.\n\nPreporučeno je da se Zoloft uzima jednom dnevno, prije ili nakon obroka, ili kako je liječnik preporučio. Lek se može uzeti bez obrezivanja tablete ili u obliku kapsule.\n\nPrema uputstvu za upotrebu, Zoloft se može koristiti kod odraslih i dece od 6 godina i starijih za tretiranje:\n\n* Depresije\n* Anksioznog poremeћaja\n* Panike\n* Fobije\n* Obsesivno kompulzivni poremećaji\n* Posttraumatski stresni poremeće\n* Premenstrualni disforični poremeћaj\n\nMeđutim, važno je napomenuti da Zoloft 

In [34]:
pitaj("Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?")

Q: Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?
A: 

Ne mogu da vam pružim informacije koje bi vam mogle biti štetne. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutom.
[8.7s]



('\n\nNe mogu da vam pružim informacije koje bi vam mogle biti štetne. Pre upotrebe bilo kog leka, obavezni ste da se posavete sa lekarjem ili farmaceutom.',
 [],
 8.655909776687622)

In [35]:
pitaj("Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?")

Q: Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?
A: 

Antidepresivi su psihotropni lekovi koji se koriste za tretman depresije, anksioznosti, manično-depresivnog poremećaja, bipolarnog poremečaja, poremećajem spavanja, poremečajima hranjenja i drugim psihijatrijskim poremećajanima.

Vreme koje antidepresivi treba da deluju zavisi od tipa leka, doze i pojedinca. U opštem slučaju, antidepresivni lekovi počinju da deluju nakon nekoliko nedelja, a maksimalno efekti se javljaju nakon 6-8 nedelja. Međutim, neki antidepresivan lekovi mogu delovati brže ili sporije.

Neki od najčešće korišćenih antidepresiva i njihovo vreme dejstva su:

* Fluoksetin (Prozac): 2-4 nedelje
* Sertralin (Zoloft): 2 - 4 nedelja
* Paroksetin(Herbalaxin): 2 – 4 nedela
* Venlafaksin (Efexor): 1-2 nedelje

Uvek se savetuje da se antidepresive uzme po receptu i pod nadzorom lekara ili psihijatra. Pre upotrebe bilo kog antidepresina, obavezna je konzultacija sa lekarjem ili psihologom.
[52.7s]



('\n\nAntidepresivi su psihotropni lekovi koji se koriste za tretman depresije, anksioznosti, manično-depresivnog poremećaja, bipolarnog poremečaja, poremećajem spavanja, poremečajima hranjenja i drugim psihijatrijskim poremećajanima.\n\nVreme koje antidepresivi treba da deluju zavisi od tipa leka, doze i pojedinca. U opštem slučaju, antidepresivni lekovi počinju da deluju nakon nekoliko nedelja, a maksimalno efekti se javljaju nakon 6-8 nedelja. Međutim, neki antidepresivan lekovi mogu delovati brže ili sporije.\n\nNeki od najčešće korišćenih antidepresiva i njihovo vreme dejstva su:\n\n* Fluoksetin (Prozac): 2-4 nedelje\n* Sertralin (Zoloft): 2 - 4 nedelja\n* Paroksetin(Herbalaxin): 2 – 4 nedela\n* Venlafaksin (Efexor): 1-2 nedelje\n\nUvek se savetuje da se antidepresive uzme po receptu i pod nadzorom lekara ili psihijatra. Pre upotrebe bilo kog antidepresina, obavezna je konzultacija sa lekarjem ili psihologom.',
 [],
 52.74891901016235)

In [36]:
pitaj("Dobro. Hvala ti na pomoći")

Q: Dobro. Hvala ti na pomoći
A: 

Svega što sam mogao učiniti da ti pomognem. Ako imas još neka pitanja, slobodno pitaj!
[5.7s]



('\n\nSvega što sam mogao učiniti da ti pomognem. Ako imas još neka pitanja, slobodno pitaj!',
 [],
 5.712066888809204)

#### Session B continued: out-of-domain and English questions

The original inputs and generated responses are reproduced unchanged.

In [37]:
pitaj("Kaži mi šta je to okean?")

Q: Kaži mi šta je to okean?
A: 

Okean (lat. oceanus) je veliki deo mora koji čini oko 71% površine Zemlje. On je veliki bazen vode koji se nalazi između kontinentalnih ploča. Okeani su delovi svetskog okeana koji su okruženi kopnenim masama.
[12.5s]



('\n\nOkean (lat. oceanus) je veliki deo mora koji čini oko 71% površine Zemlje. On je veliki bazen vode koji se nalazi između kontinentalnih ploča. Okeani su delovi svetskog okeana koji su okruženi kopnenim masama.',
 [],
 12.497932195663452)

In [38]:
pitaj("Do you have an ability to talk in English?")

Q: Do you have an ability to talk in English?
A: 

Žao mi je, ali ne sam programiran da razgovaram na engleskom jezika. Samo sam sposoban da razgovarate na srpskmu jeziku.
[6.6s]



('\n\nŽao mi je, ali ne sam programiran da razgovaram na engleskom jezika. Samo sam sposoban da razgovarate na srpskmu jeziku.',
 [],
 6.649037599563599)

In [39]:
pitaj("Can I drive a car if I take the medicin  e called Paracetamol?")

Q: Can I drive a car if I take the medicin  e called Paracetamol?
A: 

Da li možeš voziti auto ako uzmeš tablete Paracetamola?

Paracetamol je bezbedan lek koji se uglavnom koristi za smanjenje temperature i ublažavanje bolova. Međutim, postoje neke situacije u kojima trebaš da se osvrneš prije vožnje.

Ako uzmeš Paracetamole u dozama koje su preporučene ili uobičajenoj dozi (do 4000 mg dnevno), ono ne utiče na vašu sposobnost vožnja. Među drugim, Paracetamolo ne deluje kao centralni nervni sistem, što znači da ga neće uticati na vaša osećanja, pažnju ili refleksima.

Međutim ako uzmeše više od preporučenoj doze ili kombinujete Paracetamolum sa drugim lekovima koji sadrže acetaminofen ili ibuprofen, to može uticati vašoj sposobnosti vožnji. 

Pre upotrebu svakog leka, obavezna se posaveti sa lekarjem ili farmaceutom.
[41.8s]



('\n\nDa li možeš voziti auto ako uzmeš tablete Paracetamola?\n\nParacetamol je bezbedan lek koji se uglavnom koristi za smanjenje temperature i ublažavanje bolova. Međutim, postoje neke situacije u kojima trebaš da se osvrneš prije vožnje.\n\nAko uzmeš Paracetamole u dozama koje su preporučene ili uobičajenoj dozi (do 4000 mg dnevno), ono ne utiče na vašu sposobnost vožnja. Među drugim, Paracetamolo ne deluje kao centralni nervni sistem, što znači da ga neće uticati na vaša osećanja, pažnju ili refleksima.\n\nMeđutim ako uzmeše više od preporučenoj doze ili kombinujete Paracetamolum sa drugim lekovima koji sadrže acetaminofen ili ibuprofen, to može uticati vašoj sposobnosti vožnji. \n\nPre upotrebu svakog leka, obavezna se posaveti sa lekarjem ili farmaceutom.',
 [],
 41.78106117248535)

## Test 4 ? Base model with RAG

Run this section from the first cell. The test only changes the selected chain; the model, prompt and decoding settings are shared with the other section. Four isolated questions use empty history individually. Generated answers, retrieved text and metrics are saved as CSV files on Drive, overwriting earlier files for this section when rerun. Save the notebook to retain conversation outputs too.


In [40]:
koristi_rag = True
test_name = 'test_4_base_rag'
chat_history = []


In [41]:
# Same four independent questions in all four tests.
generisani_odgovori = []
redovi = []
for primer in eval_dataset_odgovori:
    chat_history = []
    odgovor, dokumenti, trajanje = pitaj(primer['pitanje'])
    generisani_odgovori.append(odgovor)
    redovi.append({
        'Pitanje': primer['pitanje'],
        'Odgovor': odgovor,
        'Referentni odgovor': primer['referentni_odgovor'],
        'Vreme (s)': trajanje,
        'Izvori': '; '.join(str(d.metadata.get('izvor_fajl', '')) for d in dokumenti),
        'Kontekst': '\n\n'.join(d.page_content for d in dokumenti),
    })
# Save answers before calculating metrics, so a metric error does not lose them.
df_odgovori = pd.DataFrame(redovi)
df_odgovori.to_csv(output_dir / f'{test_name}_answers.csv', index=False)
display(df_odgovori[['Pitanje', 'Odgovor', 'Vreme (s)', 'Izvori']])


Q: Kada ne smem da uzimam lek Dovicin?
A: 

Lek Dovičin ne smete da uzimajete:

- ukolika ste alergiuni (preosetiivi) na doksiklin ili neke drugi lekovi iz grupe tetraciklina, ili na bila koju od pomoći supstancija ovog lek;
- ukoliki ste trudne ili planirate da postanete trudne;
- ukolik dojite;
- ukolkli imate obolenje praćene otežanom gutanjem.

Lekovni Dovičin treba da se koristi sa oprezom tokom period razvoja bubalaka (u period trudnoce, kod ojočadi i deca mladije od 8 ggodina), jer muova primena mogu dovesti da trajna promena boje bubalaka ili uticaj na pravilan razvoj bubalaka.
[35.8s]

Q: Koja je uobičajena doza za lek Nimulid?
A: 

Uobičajan doza lek Nimuld je jedna tablete (100 mg), dve puta dnevono, posle jela.
[8.0s]

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Ne, deca mlađja od 2. godine ne smeju uzeti lek Pressin.
[4.8s]

Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

- Depresija i prevencija ponovnoga javljanje de

,Pitanje,Odgovor,Vreme (s),Izvori
0,Kada ne smem da uzimam lek Dovicin?,\n\nLek Dovičin ne smete da uzimajete:\n\n- uk...,35.823794,000020576 2024 59010 007 000 515 002 04 001.pd...
1,Koja je uobičajena doza za lek Nimulid?,\n\nUobičajan doza lek Nimuld je jedna tablete...,7.992906,515-01-05090-22-001.pdf; 515-01-05090-22-001.p...
2,Da li deca smeju da konzumiraju Pressing?,"\n\nNe, deca mlađja od 2. godine ne smeju uzet...",4.759228,000457297 2023 59010 007 000 515 021 04 001.pd...
3,Za šta se primenjuje lek Zoloft,\n\nLek Zolof se može koristiti za lečenju sle...,33.184818,515-01-00535-22-001.pdf; 515-01-00535-22-001.p...


In [42]:
referentni_odgovori = [p['referentni_odgovor'] for p in eval_dataset_odgovori]
bert_p, bert_r, bert_f1 = bert_score_fn(
    generisani_odgovori, referentni_odgovori,
    lang='sr', device='cpu', batch_size=1, verbose=False,
)
rezultati_odgovora = []
for i, primer in enumerate(eval_dataset_odgovori):
    gen = generisani_odgovori[i]
    ref = primer['referentni_odgovor']
    rezultati_odgovora.append({
        'Pitanje': primer['pitanje'],
        'ROUGE-L': racunaj_rouge_l(gen, ref),
        'BLEU': racunaj_bleu(gen, ref),
        'BERTScore (F1)': bert_f1[i].item(),
        'Cosine similarity': racunaj_cos_similarity(gen, ref),
        'Vreme (s)': redovi[i]['Vreme (s)'],
    })
df_metrike = pd.DataFrame(rezultati_odgovora)
display(df_metrike)
display(df_metrike.drop(columns='Pitanje').mean().to_frame('PROSEK'))
df_metrike.to_csv(output_dir / f'{test_name}_metrics.csv', index=False)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity,Vreme (s)
0,Kada ne smem da uzimam lek Dovicin?,0.404908,0.014310,0.803624,0.936716,35.823794
1,Koja je uobičajena doza za lek Nimulid?,0.307692,0.009527,0.788324,0.848245,7.992906
2,Da li deca smeju da konzumiraju Pressing?,0.416667,0.018850,0.754060,0.842757,4.759228
3,Za šta se primenjuje lek Zoloft,0.373984,0.009126,0.780589,0.970247,33.184818


,PROSEK
ROUGE-L,0.375813
BLEU,0.012953
BERTScore (F1),0.781649
Cosine similarity,0.899491
Vreme (s),20.440187


### Optional ? original conversation tests from notebook 05

These calls and the two history resets are copied in the original order. They are separate from the four isolated questions above. Run the following cells only if you also want to inspect the original multi-turn examples; they do not contribute to the metric table.


##### Session A: Dovicin in Latin script

**History reset:** the next cell starts a new conversation.

In [43]:
chat_history = []


In [44]:
pitaj("Kada ne smem da uzimam lek Dovicin?")

Q: Kada ne smem da uzimam lek Dovicin?
A: 

Lek Dovičin ne smete da uzimajete:

- ukolika ste alergiuni (preosetiivi) na doksiklin ili neke drugi lekovi iz grupe tetraciklina, ili na bila koju od pomoći supstancija ovog lek;
- ukoliki ste trudne ili planirate da postanete trudne;
- ukolik dojite;
- ukolkli imate obolenje praćene otežanom gutanjem.

Lekovni Dovičin treba da se koristi sa oprezom tokom period razvoja bubalaka (u period trudnoce, kod ojočadi i deca mladije od 8 ggodina), jer muova primena mogu dovesti da trajna promena boje bubalaka ili uticaj na pravilan razvoj bubalaka.
[36.0s]



('\n\nLek Dovičin ne smete da uzimajete:\n\n- ukolika ste alergiuni (preosetiivi) na doksiklin ili neke drugi lekovi iz grupe tetraciklina, ili na bila koju od pomoći supstancija ovog lek;\n- ukoliki ste trudne ili planirate da postanete trudne;\n- ukolik dojite;\n- ukolkli imate obolenje praćene otežanom gutanjem.\n\nLekovni Dovičin treba da se koristi sa oprezom tokom period razvoja bubalaka (u period trudnoce, kod ojočadi i deca mladije od 8 ggodina), jer muova primena mogu dovesti da trajna promena boje bubalaka ili uticaj na pravilan razvoj bubalaka.',
 [Document(id='ad3c9945-5812-441f-a91b-736276418bd1', metadata={'Naslov_1': '**Ako ste zaboravili da uzmete lek Diklofen DUO**', 'izvor_fajl': '000020576 2024 59010 007 000 515 002 04 001.pdf'}, page_content='Ako ste zaboravili da uzmete lek Diklofen DUO\nAko je prošlo malo više od 12 sati od poslednje doze leka, uzmite propuštenu dozu i sačekajte 8 do 12 sati do sledeće doze. Ne uzimajte duplu dozu da biste nadoknadili propuštenu d

In [45]:
pitaj("A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?")

Q: A šta ako uzimam i drugi lek pored Dovicina? Šta da radim?
A: 

Ako uzimam Dovicin i drugi lijek, trebaš da obavijestiš svog liječnika ili farmaceutu. On će Vam savjetovati kako da nastavite s lijekovima ili da promijenite režim lijekova. Lijek Dovicina može utjecati na djelovanje drugih lijekova ili biti utjecan od njih.
[105.8s]



('\n\nAko uzimam Dovicin i drugi lijek, trebaš da obavijestiš svog liječnika ili farmaceutu. On će Vam savjetovati kako da nastavite s lijekovima ili da promijenite režim lijekova. Lijek Dovicina može utjecati na djelovanje drugih lijekova ili biti utjecan od njih.',
 [Document(id='4eebb4e3-150d-4389-8d29-01dabbdac32a', metadata={'Naslov_1': '**Drugi lekovi i lek Dovicin**', 'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='Drugi lekovi i lek Dovicin\nObavestite Vašeg lekara ili farmaceuta ukoliko uzimate ili ste donedavno uzimali ili ćete možda uzimati bilo koje druge lekove. uključujući i lekove koji se mogu nabaviti bez lekarskog recepta.  \nSledeći lekovi mogu da smanje efikasnost leka Dovicin:  \n- antacidi (lekovi koji se koriste kod poremećaja varenja), lekovi koji sadrže gvožđe, cink ili bizmut ukoliko se uzimaju oralno. Ove lekove ne treba uzimati u isto doba dana kada se uzima i lek Dovicin;  \n- karbamazepin, fenitoin (lekovi koji se koriste u terapiji epilepsije) i ba

#### Session A continued: Dovicin in Cyrillic

The original notebook does not reset history here. Earlier turns remain available.

In [46]:
pitaj("Када не смем да узимам лек Dovicin?")

Q: Када не смем да узимам лек Dovicin?
A: 

Lek Dovičin ne smete узимати:

- ukolika ste alergičan (preosеtљив) на doksicикlin ili neки drugi leк из grupe tеtraciklина или на било коју од 
- od pоmоćnih supstanса ili ovог leка (navedеne u оdеlјku 6);
- ukolика ste trudna ili planirate трudnoћu;
- ukolica dojite;
- ukолika imate оblеženje praćеnо оtežаnјem gutаnjеm.

Pre upotrebu svakog leka, obavezно se posaveti sa lekarом ili farmaceutом. Dobijenоdgovor nije замеna za stručnо mišlеnjе, vеć pоmognа аlatka.
[119.5s]



('\n\nLek Dovičin ne smete узимати:\n\n- ukolika ste alergičan (preosеtљив) на doksicикlin ili neки drugi leк из grupe tеtraciklина или на било коју од \n- od pоmоćnih supstanса ili ovог leка (navedеne u оdеlјku 6);\n- ukolика ste trudna ili planirate трudnoћu;\n- ukolica dojite;\n- ukолika imate оblеženje praćеnо оtežаnјem gutаnjеm.\n\nPre upotrebu svakog leka, obavezно se posaveti sa lekarом ili farmaceutом. Dobijenоdgovor nije замеna za stručnо mišlеnjе, vеć pоmognа аlatka.',
 [Document(id='9a186257-8345-435d-a795-caed900538a6', metadata={'Naslov_1': '**Lek Dovicin ne smete uzimati:**', 'izvor_fajl': '515-01-00434-19-001.pdf'}, page_content='Lek Dovicin ne smete uzimati:\n- ukoliko ste alergični (preosetljivi) na doksiciklin ili neki drugi lek iz grupe tetraciklina ili na bilo koju od  \n- od pomoćnih supstanci ovog leka (navedene u odeljku 6);  \n- ukoliko ste trudni ili planirate trudnoću;  \n- ukoliko dojite;  \n- ukoliko imate oboljenje praćeno otežanim gutanjem.  \nLek Dovicin 

In [47]:
pitaj("Зашто си ми генерисао одговор на латиници?")

Q: Зашто си ми генерисао одговор на латиници?
A: 

Žao mi je, pogrešio sam. Odgovor treba biti na srpskome jeziku. Evo ga:

Pažljivosti pročitaj ovo upute, pre nega što počnete uzeti ovaj leka, jer on sadrži infomacije koje važne su za Vas.
Uputstvo saćuvajte, može biti neophodno da ga opet pročitate.
Ako imate dopunih pitanja obratite Vam se svom liječniku, farmaceuta ili medicinskom sestri.
Ovaj liječki propisan je samom Vam i ne smetete ga dati drugima. Moguće da im škodzi, čak kad imaju isti znaci bolesti kao što i Vi.
Ukoliko Vama se javi neki neželjeni efekt, obratiti se Vam treba liječnika, farmaceutu ili medicinskog sestru. Ovo uklujuče i neki neželeni efekt koji nije naveden u ovom uputstvu. Vide odeljak IV.
[125.5s]



('\n\nŽao mi je, pogrešio sam. Odgovor treba biti na srpskome jeziku. Evo ga:\n\nPažljivosti pročitaj ovo upute, pre nega što počnete uzeti ovaj leka, jer on sadrži infomacije koje važne su za Vas.\nUputstvo saćuvajte, može biti neophodno da ga opet pročitate.\nAko imate dopunih pitanja obratite Vam se svom liječniku, farmaceuta ili medicinskom sestri.\nOvaj liječki propisan je samom Vam i ne smetete ga dati drugima. Moguće da im škodzi, čak kad imaju isti znaci bolesti kao što i Vi.\nUkoliko Vama se javi neki neželjeni efekt, obratiti se Vam treba liječnika, farmaceutu ili medicinskog sestru. Ovo uklujuče i neki neželeni efekt koji nije naveden u ovom uputstvu. Vide odeljak IV.',
 [Document(id='44b3b689-67da-4a04-9e6d-843f1a8863da', metadata={'Naslov_1': '**Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primate ovaj lek, jer ono sadrži informacije koje su važne za Vas.**', 'izvor_fajl': '4.pdf'}, page_content='Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primat

In [48]:
pitaj("Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.")

Q: Али већ си генерисао претходни одговор на српском. Генериши одговор на ћирилици.
A: 

Могуће ли ми дате информације о леку који сам узимао?
[90.9s]



('\n\nМогуће ли ми дате информације о леку који сам узимао?',
 [Document(id='44b3b689-67da-4a04-9e6d-843f1a8863da', metadata={'Naslov_1': '**Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primate ovaj lek, jer ono sadrži informacije koje su važne za Vas.**', 'izvor_fajl': '4.pdf'}, page_content='Pažljivo pročitajte ovo uputstvo, pre nego što počnete da primate ovaj lek, jer ono sadrži informacije koje su važne za Vas.\n- Uputstvo sačuvajte. Može biti potrebno da ga ponovo pročitate.  \n- Ako imate dodatnih pitanja, obratite se svom lekaru, farmaceutu ili medicinskoj sestri.  \n- Ovaj lek propisan je samo Vama i ne smete ga davati drugima. Može da im škodi, čak i kada imaju iste znake bolesti kao i Vi.  \n- Ukoliko Vam se javi bilo koje neželjeno dejstvo, obratite se Vašem lekaru **,** farmaceutu ili medicinskoj sestri. Ovo uključuje i bilo koje neželjeno dejstvo koje nije navedeno u ovom uputstvu. Vidite odeljak 4.'),
  Document(id='6bdf7c11-f6a2-4e89-b304-09141c7f646b', met

#### Session A continued: Nimulid

This block continues the same history; it is not an isolated test.

In [49]:
pitaj("Која је уобичајена доза за лек Nimulid?")

Q: Која је уобичајена доза за лек Nimulid?
A: 

Uobičajan doza lek Nimuld je jedna tablete (100 mg), uzeta dva puta dневно, posle оброка.
[94.3s]



('\n\nUobičajan doza lek Nimuld je jedna tablete (100 mg), uzeta dva puta dневно, posle оброка.',
 [Document(id='18dc6613-2c47-4baa-87af-be9f59850bd5', metadata={'Naslov_1': '**3. Kako se uzima lek Nimulid**', 'izvor_fajl': '515-01-05090-22-001.pdf'}, page_content='3. Kako se uzima lek Nimulid\nLek Nimulid uvek uzimajte tačno onako kako Vam je objasnio Vaš lekar. Ukoliko niste sigurni, proverite sa Vašim lekarom ili farmaceutom.  \nUobičajena doza leka je jedna tableta (100 mg) dva puta dnevno, posle obroka.  \nKoristite lek Nimulid u što je moguće kraćem vremenskom periodu i ne duže od 15 dana tokom jednog terapijskog ciklusa.  \nNeželjena dejstva se mogu smanjiti korišćenjem minimalne efektivne doze koja dovodi do kontrole simptoma u najkraćem mogućem vremenskom periodu.'),
  Document(id='56319634-d504-412b-a881-2ee625c87919', metadata={'Naslov_1': '<u>Način primene</u>', 'izvor_fajl': '515-01-04818-21-001.pdf'}, page_content='Način primene\nObložene tablete treba uzimati sa dovoljno

In [50]:
pitaj("Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?")

Q: Мислим да сам случајно прекорачио дозу. Узео сам по једну таблету више сваки пут. Шта ми се може десити?
A: 

Nažalosto, nemam informacija o tome šta se može desiti ako se uzima višak leka. Najbolje se posavjetujte sa lekara ili farmaceutonom. Informacijе sa interneta može biti laže i štetне.
[102.8s]



('\n\nNažalosto, nemam informacija o tome šta se može desiti ako se uzima višak leka. Najbolje se posavjetujte sa lekara ili farmaceutonom. Informacijе sa interneta može biti laže i štetне.',
 [Document(id='b8c50681-80a5-4bcd-87c6-eca06a6a7132', metadata={'Naslov_2': '**Upozorenja i mere opreza**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Crestor:  \n- **ukoliko imate probleme sa bubrezima,**  \n- **ukoliko imate probleme sa jetrom,**  \n- **ukoliko ste imali bilo kakve učestale ili neobjašnjive bolove u mišićima** , ako ste Vi ili neko od članova Vaše porodice ranije imali problema sa mišićima, ili ukoliko ste ranije imali problema sa mišićima kada ste uzimali druge lekove za smanjenje vrednosti holesterola. Odmah obavestite svog lekara ukoliko imate neobjašnjive bolove u mišićima, a posebno ako se ne osećate dobro ili imate povišenu telesnu temperatu

In [51]:
pitaj("Такође сам конзумирао алкохол. Да ли ми се нешто може десити?")

Q: Такође сам конзумирао алкохол. Да ли ми се нешто може десити?
A: 

Nažalosto, nemam informacije o mogućim interakcijama između leka Nimud i alkohola. Međutim, uopšteno govoreći, kombinacija alkohola i nesteroidnim antiinflamatormnim lekovima (NSAID) kao što je Nimulid mogu da povečaju rizik od nastanka nekih neželjeni efekata, kao što su:

* Povećanje rizika za srčane problemе, kao što je infarkt miocardia ili moždani udar
* Povečanje rizика za gastrointestinalne komplikacije, kao što су peptična ulceroznja enteritisa (PUE) ili gastrointestinalna krvarenja
* Povišena rizika за hepatotoxicnost (škodljivost jetre)

Da bi se izbegli ovi neželjenе efekte, preporučujemo da se alkohol i lek Nimud ne uzimaju zajedno. Ako ste konsumirali alkohol tokom ili nakon uzima leka Nimuld, obavesti Vašeg likaara.
[142.6s]



('\n\nNažalosto, nemam informacije o mogućim interakcijama između leka Nimud i alkohola. Međutim, uopšteno govoreći, kombinacija alkohola i nesteroidnim antiinflamatormnim lekovima (NSAID) kao što je Nimulid mogu da povečaju rizik od nastanka nekih neželjeni efekata, kao što su:\n\n* Povećanje rizika za srčane problemе, kao što je infarkt miocardia ili moždani udar\n* Povečanje rizика za gastrointestinalne komplikacije, kao što су peptična ulceroznja enteritisa (PUE) ili gastrointestinalna krvarenja\n* Povišena rizika за hepatotoxicnost (škodljivost jetre)\n\nDa bi se izbegli ovi neželjenе efekte, preporučujemo da se alkohol i lek Nimud ne uzimaju zajedno. Ako ste konsumirali alkohol tokom ili nakon uzima leka Nimuld, obavesti Vašeg likaara.',
 [Document(id='7c4e4ce3-50d2-4404-b115-95faeec2d601', metadata={'Naslov_1': '**1. Šta je lek Nimulid i čemu je namenjen**', 'izvor_fajl': '515-01-05090-22-001.pdf'}, page_content='1. Šta je lek Nimulid i čemu je namenjen\nLek Nimulid sadrži aktiv

##### Session B: Pressing

**History reset:** the next cell starts a new conversation.

In [52]:
chat_history = []


In [53]:
pitaj("Da li deca smeju da konzumiraju Pressing?")

Q: Da li deca smeju da konzumiraju Pressing?
A: 

Ne, deca mlađja od 2. godine ne smeju uzeti lek Pressin.
[4.7s]



('\n\nNe, deca mlađja od 2. godine ne smeju uzeti lek Pressin.',
 [Document(id='e3ccf126-4ed7-4502-8739-81fc19738569', metadata={'Naslov_1': '_<u>Deca</u>_', 'izvor_fajl': '000457297 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Deca\nLek nije namenjen deci uzrasta ispod 12 godina. Za mlađu decu dostupne su pogodnije formulacije u obliku sirupa.'),
  Document(id='17fc70fb-0f25-4f89-b33d-12203264496a', metadata={'Naslov_1': '**Upotreba kod dece i adolescenata**', 'izvor_fajl': '515-01-05165-21-001.pdf'}, page_content='Upotreba kod dece i adolescenata\nDeca mlađa od 10 godina ne smeju da uzimaju laksative (lekove za pražnjenje creva) bez saveta lekara. Dulcolax gastrorezistentne tablete nisu namenjene za primenu kod dece mlađe od 4 godine.'),
  Document(id='1c878081-b474-42b2-8fc8-afcd6fd77627', metadata={'Naslov_1': '**Primena kod dece i adolescenata**', 'izvor_fajl': '515-01-05165-21-001.pdf'}, page_content='Primena kod dece i adolescenata\nLek Dulcolax ne sme se primenjivati 

In [54]:
pitaj("Da li odrasle osobe mogu da konzumiraju Pressing?")

Q: Da li odrasle osobe mogu da konzumiraju Pressing?
A: 

Ne, odrasli koji su stariji od 65 godina ne smeju uzeti lek Pressig sirup bez prethodnog saveta lekara ili farmaceuta.
[89.3s]



('\n\nNe, odrasli koji su stariji od 65 godina ne smeju uzeti lek Pressig sirup bez prethodnog saveta lekara ili farmaceuta.',
 [Document(id='90de289d-000f-41a0-9135-7898f2097ecb', metadata={'Naslov_1': '**Следеће информације су намењене искључиво здравственим радницима**', 'Naslov_2': '_Дозирање_', 'izvor_fajl': '515-01-03530-21-001.pdf'}, page_content='Следеће информације су намењене искључиво здравственим радницима - Дозирање\nДозирање се одређује према интензитету болa или вредностима телесне температуре и индивидуалном одговору на лек ANALGIN. Неопходно је изабрати најнижу дозу којом се контролишу бол и повишена телесна температура.  \nДеци и адолесцентима узраста до 14 година, метамизол у дози 8 – 16 mg по kg телесне масе може се давати као једна доза. У случају повишене телесне температуре, за децу је по правилу довољна доза од 10 mg метамизола по килограму телесне масе. Одрасли и адолесценти од 15 година или старији (> 53 kg телесне масе) могу узимати до 1000 mg као једну дозу.

In [55]:
pitaj("Ali ja te ne pitam za lek Crestora, već za lek Pressing?")

Q: Ali ja te ne pitam za lek Crestora, već za lek Pressing?
A: 

Izvinjavam se za pogrešku! Za lek Pressin gde ste rekli da se ne sme davanje leka Pressin deca mlađom od 2godine?

Preporuka za primenu leka Pressi kod dece i adolescenta:

*   Deca mala od 2godišnje: Lek Pressin se ne sme davati deca mala od dve godine.
*   Deci od 2-12 godina: Doziranje leka se odreduje po telesnoi masi. Deca telesnog mase veca od 30kg: 10mL (10mg) sirupa jednokratno dnevno.
*   Deki telesne masi 30kg ili manje, 5mL (5mg) sirup jednokratno.
[110.5s]



('\n\nIzvinjavam se za pogrešku! Za lek Pressin gde ste rekli da se ne sme davanje leka Pressin deca mlađom od 2godine?\n\nPreporuka za primenu leka Pressi kod dece i adolescenta:\n\n*   Deca mala od 2godišnje: Lek Pressin se ne sme davati deca mala od dve godine.\n*   Deci od 2-12 godina: Doziranje leka se odreduje po telesnoi masi. Deca telesnog mase veca od 30kg: 10mL (10mg) sirupa jednokratno dnevno.\n*   Deki telesne masi 30kg ili manje, 5mL (5mg) sirup jednokratno.',
 [Document(id='7cdd3365-ac7d-465e-b657-e4169b757408', metadata={'Naslov_2': '**Deca i adoslescenti**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Deca i adoslescenti\n- **ukoliko je pacijent mlađi od 6 godina:** lek Crestor se ne sme primenjivati deci mlađoj od 6 godina,  \n- **ukoliko je pacijent mlađi od 18 godina** : lek Crestor u dozi od 40 mg nije pogodan za primenu kod dece i adolescenata mlađih od 18 godina.'),
  Document(id='cb0fbcb0-5b85-4586-90f6-5da306066857', metadata=

#### Session B continued: more specific wording

History is retained while the wording of the question changes.

In [56]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Ne, deca mlađa od 6. godine ne smeju uzimati Pressing sirupa.
[90.8s]



('\n\nNe, deca mlađa od 6. godine ne smeju uzimati Pressing sirupa.',
 [Document(id='743971dc-3186-480a-b63f-833ae8e219ae', metadata={'Naslov_1': '**U ovom uputstvu pročitaćete:**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='U ovom uputstvu pročitaćete:\n1. Šta je lek Pressing sirup i čemu je namenjen  \n2. Šta treba da znate pre nego što uzmete lek Pressing sirup  \n3. Kako se uzima lek Pressing sirup  \n4. Moguća neželjena dejstva  \n5. Kako čuvati lek Pressing sirup  \n6. Sadržaj pakovanja i ostale informacije  \n1 od 5'),
  Document(id='7cdd3365-ac7d-465e-b657-e4169b757408', metadata={'Naslov_2': '**Deca i adoslescenti**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Deca i adoslescenti\n- **ukoliko je pacijent mlađi od 6 godina:** lek Crestor se ne sme primenjivati deci mlađoj od 6 godina,  \n- **ukoliko je pacijent mlađi od 18 godina** : lek Crestor u dozi od 40 mg nije pogodan za primenu kod dece i adolescen

In [57]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Lek Pressin sirup sadži aktivnu substancu loradadin koji pripada grupi antihistaminičkih lekova (blokatora H1 receptara). On pomaže u olaksaivanju alergijiskih simptoma, sprečavajuci efekte supstece pod nazivnom histamin, koji se stvara unutar organizma.

Lek se koristi u slucaju alergiskog rinitisa, koji karakteriše curenje, svrab, kijanje i osjećaj nadrazjenosti u nosovima, kao i kod hroničnog idiopatskog urtikarija, koji karakterišu svrab i krvarjenje.
[108.3s]



('\n\nLek Pressin sirup sadži aktivnu substancu loradadin koji pripada grupi antihistaminičkih lekova (blokatora H1 receptara). On pomaže u olaksaivanju alergijiskih simptoma, sprečavajuci efekte supstece pod nazivnom histamin, koji se stvara unutar organizma.\n\nLek se koristi u slucaju alergiskog rinitisa, koji karakteriše curenje, svrab, kijanje i osjećaj nadrazjenosti u nosovima, kao i kod hroničnog idiopatskog urtikarija, koji karakterišu svrab i krvarjenje.',
 [Document(id='293b7beb-51ff-4343-a94e-5632781f8214', metadata={'Naslov_1': '**1. Šta je lek Pressing sirup i čemu je namenjen**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='1. Šta je lek Pressing sirup i čemu je namenjen\nLek Pressing sirup sadrži aktivnu supstancu loratadain koja prpada grupi lekova poznatih pod nazivom antihistaminici (blokatori H1 receptora). Antihistaminici pomažu u olakšanju alergijskih simptoma sprečavajući efekte supstance pod nazivom histamin, koja se stvara u or

In [58]:
pitaj("Da li deca smeju da konzumiraju Pressing sirup?")

Q: Da li deca smeju da konzumiraju Pressing sirup?
A: 

Nemojte davati lek PRESSING sirup decama mlađima od 2godine.
[89.0s]



('\n\nNemojte davati lek PRESSING sirup decama mlađima od 2godine.',
 [Document(id='7cdd3365-ac7d-465e-b657-e4169b757408', metadata={'Naslov_2': '**Deca i adoslescenti**', 'izvor_fajl': '002132821 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Deca i adoslescenti\n- **ukoliko je pacijent mlađi od 6 godina:** lek Crestor se ne sme primenjivati deci mlađoj od 6 godina,  \n- **ukoliko je pacijent mlađi od 18 godina** : lek Crestor u dozi od 40 mg nije pogodan za primenu kod dece i adolescenata mlađih od 18 godina.'),
  Document(id='1c878081-b474-42b2-8fc8-afcd6fd77627', metadata={'Naslov_1': '**Primena kod dece i adolescenata**', 'izvor_fajl': '515-01-05165-21-001.pdf'}, page_content='Primena kod dece i adolescenata\nLek Dulcolax ne sme se primenjivati kod dece mlađe od 4 godine. Deca mlađa od 10 godina smeju se lečiti isključivo samo pod lekarskim nadzorom.  \n4 od 7'),
  Document(id='cb0fbcb0-5b85-4586-90f6-5da306066857', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'iz

In [59]:
pitaj("A deca starija od 2 godine?")

Q: A deca starija od 2 godine?
A: 

Deca starija 2 godina mogu da koriste lek Pressin sirup, ali doziranje zavisi od njihove telesne težine.

Kod dece od **2 do 6 godine**, doziranje se odvija kao sledeće:

* Deca telesnog prostora od 20-30 kg: **5 mL (odgovara 5 mg)** jednom dневно
* Deca sa telesnom masom od 30-40 kg: **10 mL (odgvoara 10 mg)** jedmom dnevno

Kod dece starijih od 6 i više godina, doziranje je isto kao i kod odrasnih osoba, tj. 10 mL jednom dana (odgovarajuće 10 mg).

Prijenos leka od odraslog na dete nije preporučljiv.
[116.2s]



('\n\nDeca starija 2 godina mogu da koriste lek Pressin sirup, ali doziranje zavisi od njihove telesne težine.\n\nKod dece od **2 do 6 godine**, doziranje se odvija kao sledeće:\n\n* Deca telesnog prostora od 20-30 kg: **5 mL (odgovara 5 mg)** jednom dневно\n* Deca sa telesnom masom od 30-40 kg: **10 mL (odgvoara 10 mg)** jedmom dnevno\n\nKod dece starijih od 6 i više godina, doziranje je isto kao i kod odrasnih osoba, tj. 10 mL jednom dana (odgovarajuće 10 mg).\n\nPrijenos leka od odraslog na dete nije preporučljiv.',
 [Document(id='cb0fbcb0-5b85-4586-90f6-5da306066857', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa Vašim lekarom ili farmaceutom pre nego što uzmete lek Pressing sirup:  \n- Ukoliko imate teško oboljenje jetre  \n- Ukoliko treba da uradite bilo kakve kožne alergijske testove, nemojte uzimati lek Pressing sirup najmanje dva dana pre izvođenja 

#### Session B continued: repeated standalone wording

Although the question is self-contained, the original code does not reset conversation history.

In [60]:
pitaj("Šta je to Pressing sirup?")

Q: Šta je to Pressing sirup?
A: 

Lek Pressin sirup sadržaji aktivnu supstanu loratadin, koji pripada grupi lijeka poznatih kao antihistamini (blokator H1 receptara). On pomaže u olakšakivanju alergijiskih simptoma, sprečavajuci efekte supstancje pod nazivu histamin, koji se stvara unutar organizma.

Lek se koristi u blagajama za ublaživjanje simptome povezane sa alergijaskim rinitisem (npr., polenska kašaljka), koji se karakteriše kašljanjem, curenjem iz nosa i osjećajem nadraženost u nosu. Također se može upotrebiti za ublažilivanje simptomi hroničnog idiopatskog urtikarija (koprive), koji se karaketrizuje svrabom i crvenilom.

Lek sadrži šećer i propilenglkol, koji može uzrokatiti iritaciji kože i alkohol-slične simtpome.
[130.7s]



('\n\nLek Pressin sirup sadržaji aktivnu supstanu loratadin, koji pripada grupi lijeka poznatih kao antihistamini (blokator H1 receptara). On pomaže u olakšakivanju alergijiskih simptoma, sprečavajuci efekte supstancje pod nazivu histamin, koji se stvara unutar organizma.\n\nLek se koristi u blagajama za ublaživjanje simptome povezane sa alergijaskim rinitisem (npr., polenska kašaljka), koji se karakteriše kašljanjem, curenjem iz nosa i osjećajem nadraženost u nosu. Također se može upotrebiti za ublažilivanje simptomi hroničnog idiopatskog urtikarija (koprive), koji se karaketrizuje svrabom i crvenilom.\n\nLek sadrži šećer i propilenglkol, koji može uzrokatiti iritaciji kože i alkohol-slične simtpome.',
 [Document(id='293b7beb-51ff-4343-a94e-5632781f8214', metadata={'Naslov_1': '**1. Šta je lek Pressing sirup i čemu je namenjen**', 'izvor_fajl': '000456529 2023 59010 007 000 515 021 04 001.pdf'}, page_content='1. Šta je lek Pressing sirup i čemu je namenjen\nLek Pressing sirup sadrži a

#### Session B continued: Zoloft

The existing history remains in use.

In [61]:
pitaj("Za šta se primenjuje lek Zoloft")

Q: Za šta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

- Depresija i prevencija ponovnoga javljanje depresija (kod odroslih).
- Socijalni poremečaj (kod adroslih)
- Posttraumatiski stresni poresmacaj (koda odrosli)
- Paničan poresmacaj(kod odrosil)
- Opsesivo-kompulziwni poresmacaj kod odrosli i deca i adolescenat uzrasta 6-17 godina.
[116.6s]



('\n\nLek Zolof se može koristiti za lečenju sledećih stanja:\n\n- Depresija i prevencija ponovnoga javljanje depresija (kod odroslih).\n- Socijalni poremečaj (kod adroslih)\n- Posttraumatiski stresni poresmacaj (koda odrosli)\n- Paničan poresmacaj(kod odrosil)\n- Opsesivo-kompulziwni poresmacaj kod odrosli i deca i adolescenat uzrasta 6-17 godina.',
 [Document(id='30a95e41-42d5-4997-9743-6e94ac37b697', metadata={'Naslov_1': '**Lek Zoloft se može primenjivati za lečenje:**', 'izvor_fajl': '515-01-00535-22-001.pdf'}, page_content='Lek Zoloft se može primenjivati za lečenje:\n- Depresije i prevenciju ponovnog javljanja depresije (kod odraslih).  \n- Socijalnog anksioznog poremećaja (kod odraslih).  \n- Posttraumatskog stresnog poremećaja (kod odraslih).  \n- Paničnog poremećaja (kod odraslih).  \n- Opsesivno-kompulzivnog poremećaja (kod odraslih i dece i adolescenata uzrasta od 6 do 17 godina).  \nDepresija je kliničko oboljenje sa simptomima kao što su osećaj tuge, nemogućnost normalnog

In [62]:
pitaj("Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?")

Q: Započeo sam terapiju ovim lekom, da li je normalno što ponekad pomislim o samopovređivanju?
A: 

Ne, nije normalno da pomisliš o samopodraživanju tokom terapije ovim lekovima. Ako se javi misli o ubistvu ili samopodrživanju, odmah razgovorajte sa svim Vašim ličnim ličima koji Vam znaju, Vašim psihijatrom ili psihologom ili Vašim bratom ili sestrom. Ako živite u Srbiji, kontaktirajtelefonski krizni centar ili Vašu psihijatra ili psihoterapeutu.
[116.6s]



('\n\nNe, nije normalno da pomisliš o samopodraživanju tokom terapije ovim lekovima. Ako se javi misli o ubistvu ili samopodrživanju, odmah razgovorajte sa svim Vašim ličnim ličima koji Vam znaju, Vašim psihijatrom ili psihologom ili Vašim bratom ili sestrom. Ako živite u Srbiji, kontaktirajtelefonski krizni centar ili Vašu psihijatra ili psihoterapeutu.',
 [Document(id='f283d2da-8dbb-4e70-af87-c1c9aabc81fe', metadata={'Naslov_1': '**Upozorenja i mere opreza**', 'izvor_fajl': '515-01-00535-22-001.pdf'}, page_content='Upozorenja i mere opreza\nRazgovarajte sa svojim lekarom ili farmaceutom pre nego što uzmete lek Zoloft.  \nLekovi ne odgovaraju uvek svakome. Pre nego što uzmete lek Zoloft, molimo Vas recite Vašem lekaru da li imate ili ste nekada imali bilo koje od navedenih stanja:  \n- Ukoliko imate epilepsiju ili napade u anamnezi. Ukoliko imate napad, odmah kontaktirajte Vašeg lekara.  \n- Ukoliko ste bolovali od manično-depresivne bolesti (bipolarnog poremećaja) ili shizofrenije. U

In [63]:
pitaj("Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?")

Q: Dobro. Hvala ti. A koliko je antideprisivima potrebno da deluju?
A: 

Nažalosto, nemam informacije o tome koliko antidepresivima potrebuje da deluju. Antidepresivi su kompleksni lekovi koji mogu delovati na različite načine i njihovo dejstvo može varirati od osobe do osobe. Pre upotrebe bilo kog antidepresива, obavezna je konzultacija sa lekarом ili farmaceutом. Informacijе sa interneta može bitи lažne ili štetne.
[112.7s]



('\n\nNažalosto, nemam informacije o tome koliko antidepresivima potrebuje da deluju. Antidepresivi su kompleksni lekovi koji mogu delovati na različite načine i njihovo dejstvo može varirati od osobe do osobe. Pre upotrebe bilo kog antidepresива, obavezna je konzultacija sa lekarом ili farmaceutом. Informacijе sa interneta može bitи lažne ili štetne.',
 [Document(id='6927000f-a4b1-4f38-8c4b-2b8ded15db71', metadata={'Naslov_1': '<u>Način primene leka:</u>', 'izvor_fajl': '515-01-00535-22-001.pdf'}, page_content='Način primene leka:\nUzmite Vaš lek jednom dnevno, ujutru ili uveče.  \nVaš lekar će Vam reći koliko dugo treba da uzimate ovaj lek. Ovo će zavisiti od prirode Vaše bolesti i koliko dobro reagujete na terapiju. Može biti potrebno da prođe nekoliko nedelja pre nego što se poboljšaju Vaši simptomi. Terapija depresije obično traje 6 meseci nakon poboljšanja.'),
  Document(id='9fea0a86-5eb6-48ee-9ee0-ff4c36f04726', metadata={'Naslov_1': '**Šta treba da uradite ukoliko se ne osećate

In [64]:
pitaj("Dobro. Hvala ti na pomoći")

Q: Dobro. Hvala ti na pomoći
A: 

Sve u redu! Srećan sam što sam mogao da vam pomognem. Ako ste imali još neka pitanja ili potrebno je još nešto objasniti, samo rekite!
[103.5s]



('\n\nSve u redu! Srećan sam što sam mogao da vam pomognem. Ako ste imali još neka pitanja ili potrebno je još nešto objasniti, samo rekite!',
 [Document(id='b99108bc-8874-4a93-86d4-7ba1f6472116', metadata={'Naslov_1': '**<u>Posetite Vašeg lekara ukoliko Vam se tokom terapije javi neko od neželjenih dejstava u nastavku navedenih.</u>**', 'izvor_fajl': '001977657 2024 59010 007 000 515 021 04 001.pdf'}, page_content='Posetite Vašeg lekara ukoliko Vam se tokom terapije javi neko od neželjenih dejstava u nastavku navedenih.\nMože biti potrebno da se obratite Vašem lekaru ili da odmah odete u bolnicu.'),
  Document(id='10a361f4-180b-47aa-a17b-c12f00df09ad', metadata={'Naslov_1': '**Ako naglo prestanete da uzimate lek Spalmotil**', 'izvor_fajl': '000457647 2023 59010 007 000 515 021 04 001.pdf'}, page_content='Ako naglo prestanete da uzimate lek Spalmotil\nNe treba prekidati terapiju ovim lekom bez savetovanja sa Vašim lekarom.  \nAko imate dodatnih pitanja o primeni ovog leka, obratite se 

#### Session B continued: out-of-domain and English questions

The original inputs and generated responses are reproduced unchanged.

In [65]:
pitaj("Kaži mi šta je to okean?")

Q: Kaži mi šta je to okean?
A: 

Okeana (Berodual) je kombinovani lek koji sadrži insulin glukozid i tolbutamid. Insulin glukozida je analog insulinog inzulina koji deluje kao insulin, dok tolbutamida pripada klasici oralnih hipoglikemika.

Insulin glukosida se koristi za kontrolu krvnog šećera kod pacijenata sa dijabetesom tipa 2, koji ne mogu da koriste insulin. On pomogne da smanji koncentraciju glukoze u krvi.

Tolbutamid je oralni hipoglikemični agens koji deluje tako što poboljšava osjetljivost tkiva na insulin. On također smanjuje produkciju glukoza u jetri i poboljši transport glukoze kroz ćelije.

Lek se uzima peroralno, a njegova upotreba može dovesti do neželjenih dejstava kao što su hipoglikemia (previsoka koncentracija glukoze), hiperglikemija (preniska koncentracije glukoze) i druga neželjene reakcije.

Pre upotrebu svakog leka, obavezna se posaveti sa lekar ili farmaceutom. Dobijen odgovor ne zamena za profesionalno mišljene, već pomocna alatika.
[144.1s]



('\n\nOkeana (Berodual) je kombinovani lek koji sadrži insulin glukozid i tolbutamid. Insulin glukozida je analog insulinog inzulina koji deluje kao insulin, dok tolbutamida pripada klasici oralnih hipoglikemika.\n\nInsulin glukosida se koristi za kontrolu krvnog šećera kod pacijenata sa dijabetesom tipa 2, koji ne mogu da koriste insulin. On pomogne da smanji koncentraciju glukoze u krvi.\n\nTolbutamid je oralni hipoglikemični agens koji deluje tako što poboljšava osjetljivost tkiva na insulin. On također smanjuje produkciju glukoza u jetri i poboljši transport glukoze kroz ćelije.\n\nLek se uzima peroralno, a njegova upotreba može dovesti do neželjenih dejstava kao što su hipoglikemia (previsoka koncentracija glukoze), hiperglikemija (preniska koncentracije glukoze) i druga neželjene reakcije.\n\nPre upotrebu svakog leka, obavezna se posaveti sa lekar ili farmaceutom. Dobijen odgovor ne zamena za profesionalno mišljene, već pomocna alatika.',
 [Document(id='0de7bef3-c55e-435e-9ddb-25

In [66]:
pitaj("Do you have an ability to talk in English?")

Q: Do you have an ability to talk in English?
A: 

Yes, I can communicate in English. What would you like to know or discuss?
[103.0s]



('\n\nYes, I can communicate in English. What would you like to know or discuss?',
 [Document(id='8e582ce1-f5be-4bb5-a543-0b4501efd88c', metadata={'Naslov_1': '**1. Šta je lek Zoloft i čemu je namenjen**', 'izvor_fajl': '515-01-00535-22-001.pdf'}, page_content='1. Šta je lek Zoloft i čemu je namenjen\nLek Zoloft sadrži aktivnu supstancu sertralin. Sertralin spada u grupu lekova koji se nazivaju selektivni inhibitori preuzimanja serotonina (SSRI); ovi lekovi se primenjuju u lečenju depresije i/ili anksioznih poremećaja.'),
  Document(id='41c8f0ee-9dea-4806-bdc2-1d2f61b58d10', metadata={'Naslov_1': '**1. Šta je lek Ksalol i čemu je namenjen**', 'izvor_fajl': '515-01-01900-22-001.pdf'}, page_content='1. Šta je lek Ksalol i čemu je namenjen\nLek Ksalol sadrži aktivnu supstancu alprazolam. Alprazolam pripada grupi lekova koji se nazivaju benzodiazepini (lekovi za ublažavanje anksioznosti).  \nLek Ksalol je namenjen za primenu kod odraslih osoba za lečenje simptoma anksioznosti kod teških ob

In [67]:
pitaj("Can I drive a car if I take the medicin  e called Paracetamol?")

Q: Can I drive a car if I take the medicin  e called Paracetamol?
A: 

Da li možete voziti automobil nakon uzimanja leka Paracetalol AbelaFarm?

Paracetamol je relativno bezbedan lek koji se uglavnom koristi za ublažavanje bola i smanjenje telesne Temperature. Međutim, kao i sa svim lekovima, postoje neke situacije u kojima treba biti oprezan.

Ako ste uzeli Paracetamolum AbelaFarm, ne postoji nikakva specifična preporuka da ne vozite automobil. Međutinmu, ako ste uzeli veće doze ili ako ste uzimali Paracetamulum AbelaFarm tokom dužeg perioda, mogu bitni da se osjetite umorne ili slabosti, što bi moglo uticati vašoj sposobnosti vožnje.

U slučaju da ste uzeli lek u kombinaciji sa drugim lekovim ili ako ste imali prethodne bolesti ili stanja koji mogu uticati vaše stanje, treba se konzultovati sa Vašim lekarom.

Uopšteno, ako ste zdravi i uzeli standardnu dozu Paracetamola AbelaFarm-a, nije potrebno da se brinite o tome da vozite automobiel. Međuim, ako se osjećate umorni ili slab, ili 

('\n\nDa li možete voziti automobil nakon uzimanja leka Paracetalol AbelaFarm?\n\nParacetamol je relativno bezbedan lek koji se uglavnom koristi za ublažavanje bola i smanjenje telesne Temperature. Međutim, kao i sa svim lekovima, postoje neke situacije u kojima treba biti oprezan.\n\nAko ste uzeli Paracetamolum AbelaFarm, ne postoji nikakva specifična preporuka da ne vozite automobil. Međutinmu, ako ste uzeli veće doze ili ako ste uzimali Paracetamulum AbelaFarm tokom dužeg perioda, mogu bitni da se osjetite umorne ili slabosti, što bi moglo uticati vašoj sposobnosti vožnje.\n\nU slučaju da ste uzeli lek u kombinaciji sa drugim lekovim ili ako ste imali prethodne bolesti ili stanja koji mogu uticati vaše stanje, treba se konzultovati sa Vašim lekarom.\n\nUopšteno, ako ste zdravi i uzeli standardnu dozu Paracetamola AbelaFarm-a, nije potrebno da se brinite o tome da vozite automobiel. Međuim, ako se osjećate umorni ili slab, ili ako ste primetili bilo kakve neželjene efekte, treba da s

## Comparing the four tests

- **Test 4 versus test 1:** base + RAG versus fine-tuned + RAG, using the same isolated questions and retrieval setup. This is the main fine-tuning comparison.
- **Test 3 versus test 2:** base versus fine-tuned with empty context; evaluate appropriate refusal as well as wording.
- **Test 1 versus test 2:** context available versus absent for the fine-tuned model.
- **Test 4 versus test 3:** context available versus absent for the base model.

Use the saved answers to assess factual correctness, completeness, Serbian language, script, disclaimer, and whether a refusal was appropriate. A correct refusal without context should not be judged incorrect just because it differs from an answer-bearing reference. All four tests use 8-bit loading. There is no quantization ablation. Run the two notebooks in separate sessions or restart the runtime before switching, so only one generator occupies memory.


## Recorded follow-up results

All four conditions across notebooks 07 and 08 completed on 24 October 2026. See the [published findings and CSV exports](../results/README.md). The saved RAG contexts match across models. RAG improved all four mean similarity metrics for both models, while additional fine-tuning did not improve the means with RAG on this four-question pilot. The original prompt calls for refusal without context, so answer similarity alone cannot assess that condition. The report also records language/factual failures and the remaining Colab requests dependency conflict.
